# TCGA clinical and expression models

Adapted from the submitted repository, commit `fce9ade29ef6de731407a7fcf14098f30a272cd3`, for the public-data workflow. Current definitions and figure mappings are documented in `docs/`.


In [ ]:
import os
from pathlib import Path
from IPython.display import display
_candidates = [Path.cwd(), *Path.cwd().parents]
_project = next((p for p in _candidates if (p / "src/mitoxy/data.py").is_file()), None)
if _project is None:
    raise RuntimeError("Launch this notebook within the Mitoxy repository")
os.environ["MITOXY_PROJECT_ROOT"] = str(_project)


# Figure 2 analysis and validation

This notebook reconstructs the purity-adjusted models, high-purity
sensitivity analysis, locked 10,000-module matched-null test, and
global-expression PC-adjusted specificity analysis used in Figure 2. It saves
compact plotting tables and full audit outputs for the plotting notebook.

All inputs are resolved from the paper-wide public-source manifest written by
`00_Paper_Data_Download.ipynb`. This notebook performs no downloads and does
not read archived derived results.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import norm, rankdata, spearmanr
from sklearn.decomposition import PCA

PROJECT_ROOT = Path(
    os.environ["MITOXY_PROJECT_ROOT"]
)
RAW_MANIFEST_PATH = (
    PROJECT_ROOT / "data" / "provenance" / "paper_public_source_manifest.csv"
)
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "figure_2_clean"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENES = ["BAX", "BAK1", "BID", "SLC7A11"]
SEED = 20260913
N_ITER = 10_000
POOL_N = 250
PURITY_THRESHOLD = 0.56

raw_manifest = pd.read_csv(RAW_MANIFEST_PATH)


def manifest_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def source_path(dataset, role):
    matches = raw_manifest.loc[
        raw_manifest["dataset"].eq(dataset)
        & raw_manifest["role"].eq(role)
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one public source for ({dataset!r}, {role!r}); "
            f"found {len(matches)}"
        )
    row = matches.iloc[0]
    path = PROJECT_ROOT / str(row["relative_path"])
    if not path.exists():
        raise FileNotFoundError(path)
    if path.stat().st_size != int(row["size_bytes"]):
        raise RuntimeError(f"Source-size mismatch: {path}")
    if manifest_sha256(path) != str(row["sha256"]):
        raise RuntimeError(f"Source SHA-256 mismatch: {path}")
    return path


RAW = {
    "tcga_xena_expression": source_path(
        "TCGA-KIRC", "tumor RNA expression"
    ),
    "tcga_gdc_clinical_json": source_path(
        "TCGA-KIRC", "clinical API response"
    ),
    "tcga_absolute_purity": source_path(
        "TCGA-KIRC", "ABSOLUTE purity"
    ),
}

print("Paper-wide public-source manifest loaded:", RAW_MANIFEST_PATH)
display(pd.DataFrame({"analysis_key": RAW.keys(), "path": RAW.values()}))


## Helpers


In [ ]:
def parse_grade(value):
    match = re.search(r"G([1-4])", str(value).upper())
    return float(match.group(1)) if match else np.nan


def parse_stage(value):
    match = re.search(r"STAGE\s+([IV]+)", str(value).upper())
    if not match:
        return np.nan
    return {"I": 1.0, "II": 2.0, "III": 3.0, "IV": 4.0}.get(
        match.group(1), np.nan
    )


def first_numeric(records, key, parser):
    for record in records:
        parsed = parser(record.get(key))
        if pd.notna(parsed):
            return parsed
    return np.nan


def zscore(values):
    values = pd.Series(values, index=getattr(values, "index", None), dtype=float)
    return (values - values.mean()) / values.std(ddof=0)


def fit_hc3(data, endpoint, analysis, purity_threshold=np.nan):
    columns = [endpoint, "mitoxy", "age", "male", "purity"]
    model_data = data[columns].dropna().copy()
    if pd.notna(purity_threshold):
        model_data = model_data.loc[
            model_data["purity"] >= purity_threshold
        ].copy()

    # The archived analysis standardized Mitoxy inside each analysis set.
    model_data["mitoxy_z"] = zscore(model_data["mitoxy"])
    design = sm.add_constant(
        model_data[["mitoxy_z", "age", "male", "purity"]]
    )
    fit = sm.OLS(model_data[endpoint], design).fit(cov_type="HC3")
    beta = float(fit.params["mitoxy_z"])
    se = float(fit.bse["mitoxy_z"])
    p_value = float(fit.pvalues["mitoxy_z"])
    return {
        "endpoint": "Grade" if endpoint == "grade" else "Stage",
        "analysis": analysis,
        "n": len(model_data),
        "purity_threshold": purity_threshold,
        "beta": beta,
        "standard_error": se,
        "ci_low": beta - norm.ppf(0.975) * se,
        "ci_high": beta + norm.ppf(0.975) * se,
        "p_value": p_value,
    }


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


## Build the TCGA-KIRC analysis table


In [ ]:
from mitoxy.clinical import primary_diagnoses
expression = pd.read_csv(
    RAW["tcga_xena_expression"], sep="\t", index_col=0
)
expression.index = expression.index.astype(str).str.upper()
missing = sorted(set(GENES) - set(expression.index))
assert not missing, missing

primary_columns = [
    column for column in expression.columns
    if len(column) >= 15 and column[13:15] == "01"
]
assert len(primary_columns) == 533

target_expression = expression.loc[GENES, primary_columns].astype(float)
target_z = target_expression.sub(
    target_expression.mean(axis=1), axis=0
).div(target_expression.std(axis=1, ddof=0), axis=0)

tcga = pd.DataFrame({
    "sample_id": [column[:12] for column in primary_columns],
    "mitoxy": target_z.mean(axis=0).to_numpy(),
})

clinical_object = json.loads(RAW["tcga_gdc_clinical_json"].read_text())
if isinstance(clinical_object, list):
    cases = clinical_object
elif "data" in clinical_object:
    cases = clinical_object["data"].get("hits", clinical_object["data"])
else:
    cases = clinical_object.get("hits", clinical_object)

clinical_rows = []
for case in cases:
    if not isinstance(case, dict):
        continue
    diagnoses = primary_diagnoses(case)
    demographic = case.get("demographic") or {}
    age = demographic.get("age_at_index")
    if age is None and demographic.get("days_to_birth") is not None:
        age = -float(demographic["days_to_birth"]) / 365.25
    sex = str(
        demographic.get("sex_at_birth", demographic.get("gender", ""))
    ).lower()
    male = 1.0 if sex == "male" else 0.0 if sex == "female" else np.nan
    clinical_rows.append({
        "sample_id": case.get("submitter_id"),
        "grade": first_numeric(diagnoses, "tumor_grade", parse_grade),
        "stage": first_numeric(
            diagnoses, "ajcc_pathologic_stage", parse_stage
        ),
        "age": pd.to_numeric(age, errors="coerce"),
        "male": male,
    })

clinical = pd.DataFrame(clinical_rows).drop_duplicates("sample_id")

purity = pd.read_csv(RAW["tcga_absolute_purity"], sep="\t")
purity["sample_id"] = purity["array"].astype(str).str.slice(0, 12)
purity = (
    purity[["sample_id", "purity"]]
    .dropna(subset=["sample_id"])
    .drop_duplicates("sample_id")
)

tcga = (
    tcga.merge(clinical, on="sample_id", how="left", validate="one_to_one")
    .merge(purity, on="sample_id", how="left", validate="one_to_one")
)

print("TCGA primary tumors:", len(tcga))
print(tcga[["grade", "stage", "age", "male", "purity"]].notna().sum())
print("Median ABSOLUTE purity:", tcga["purity"].median())

assert len(tcga) == 533
assert tcga["grade"].notna().sum() == 525
assert tcga["stage"].notna().sum() == 530
assert np.isclose(tcga["purity"].median(), PURITY_THRESHOLD)

tcga.to_csv(OUTPUT_DIR / "tcga_kirc_figure_2_analysis_table.csv", index=False)


## Panels A and B — purity-adjusted regression


In [ ]:
panel_ab = pd.DataFrame([
    fit_hc3(tcga, "grade", "Purity-adjusted full cohort"),
    fit_hc3(tcga, "stage", "Purity-adjusted full cohort"),
    fit_hc3(tcga, "grade", "High-purity sensitivity", PURITY_THRESHOLD),
    fit_hc3(tcga, "stage", "High-purity sensitivity", PURITY_THRESHOLD),
])

n_contract = {
    ("Purity-adjusted full cohort", "Grade"): 490,
    ("Purity-adjusted full cohort", "Stage"): 495,
    ("High-purity sensitivity", "Grade"): 246,
    ("High-purity sensitivity", "Stage"): 251,
}
observed_n = panel_ab.set_index(["analysis", "endpoint"])["n"].to_dict()
assert observed_n == n_contract
assert np.isfinite(
    panel_ab[["beta", "standard_error", "ci_low", "ci_high", "p_value"]]
).all().all()
assert (panel_ab["standard_error"] > 0).all()
assert (panel_ab["ci_low"] <= panel_ab["beta"]).all()
assert (panel_ab["beta"] <= panel_ab["ci_high"]).all()
assert panel_ab["p_value"].between(0, 1).all()

panel_ab.to_csv(OUTPUT_DIR / "panels_a_b_purity_models.csv", index=False)
print("Panels A/B structural audit: PASS")


## Panel C — locked matched-null specificity


In [ ]:
X = expression[primary_columns].astype("float64")
means = X.mean(axis=1)
sds = X.std(axis=1, ddof=0)
gene_stats = (
    pd.DataFrame({"mean": means, "sd": sds})
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)
gene_stats = gene_stats.loc[gene_stats["sd"] > 0].copy()
for feature in ["mean", "sd"]:
    gene_stats[f"{feature}_z"] = zscore(gene_stats[feature])

target_set = set(GENES)
pools = {}
for gene in GENES:
    target = gene_stats.loc[gene]
    distance = np.sqrt(
        (gene_stats["mean_z"] - target["mean_z"]) ** 2
        + (gene_stats["sd_z"] - target["sd_z"]) ** 2
    )
    distance = distance.drop(
        labels=[item for item in target_set if item in distance.index]
    )
    selected = distance.sort_values(kind="mergesort").head(POOL_N)
    pools[gene] = selected.index.tolist()
    pd.DataFrame({
        "gene": selected.index,
        "distance": selected.values,
    }).to_csv(OUTPUT_DIR / f"locked_pool_{gene}.csv", index=False)

needed_genes = sorted(target_set.union(*[set(values) for values in pools.values()]))
selected_expression = X.loc[needed_genes]
selected_z = selected_expression.sub(
    selected_expression.mean(axis=1), axis=0
).div(selected_expression.std(axis=1, ddof=0), axis=0)
gene_to_index = {gene: index for index, gene in enumerate(needed_genes)}
z_array = selected_z.to_numpy()

rng = np.random.default_rng(SEED)
modules = []
while len(modules) < N_ITER:
    draw = [rng.choice(pools[gene]) for gene in GENES]
    if len(set(draw)) == len(GENES):
        modules.append(draw)

module_indices = np.array([
    [gene_to_index[gene] for gene in module]
    for module in modules
], dtype=np.int32)
target_indices = np.array([gene_to_index[gene] for gene in GENES])

patient_to_column = {
    column[:12]: index for index, column in enumerate(primary_columns)
}
tcga_ordered = tcga.loc[tcga["sample_id"].isin(patient_to_column)].copy()
column_indices = np.array([
    patient_to_column[patient] for patient in tcga_ordered["sample_id"]
])

rows = []
for endpoint in ["grade", "stage"]:
    mask = tcga_ordered[endpoint].notna().to_numpy()
    endpoint_columns = column_indices[mask]
    outcome = tcga_ordered.loc[mask, endpoint].to_numpy()
    observed_score = z_array[target_indices][:, endpoint_columns].mean(axis=0)
    observed_rho = float(spearmanr(observed_score, outcome).statistic)

    outcome_rank = rankdata(outcome)
    outcome_centered = outcome_rank - outcome_rank.mean()
    outcome_denominator = np.sqrt(np.sum(outcome_centered ** 2))
    null = np.empty(N_ITER)

    for start in range(0, N_ITER, 1000):
        stop = min(start + 1000, N_ITER)
        module_scores = z_array[module_indices[start:stop]][
            :, :, endpoint_columns
        ].mean(axis=1)
        ranks = rankdata(module_scores, axis=1)
        centered = ranks - ranks.mean(axis=1, keepdims=True)
        null[start:stop] = (
            centered @ outcome_centered
        ) / (
            np.sqrt(np.sum(centered ** 2, axis=1)) * outcome_denominator
        )

    percentile = float(100 * np.mean(null < observed_rho))
    empirical_p = float((1 + np.sum(null >= observed_rho)) / (N_ITER + 1))
    rows.append({
        "endpoint": "Grade" if endpoint == "grade" else "Stage",
        "n": int(mask.sum()),
        "observed_rho": observed_rho,
        "null_mean": float(null.mean()),
        "null_sd": float(null.std(ddof=1)),
        "percentile": percentile,
        "empirical_p": empirical_p,
        "seed": SEED,
        "pool_size": POOL_N,
        "iterations": N_ITER,
    })
    pd.DataFrame({
        "rho": null,
        "genes": ["|".join(module) for module in modules],
    }).to_csv(OUTPUT_DIR / f"locked_null_{endpoint}.csv", index=False)

panel_c = pd.DataFrame(rows)
pd.DataFrame(modules, columns=GENES).to_csv(
    OUTPUT_DIR / "locked_modules.csv", index=False
)

assert panel_c.set_index("endpoint")["n"].to_dict() == {
    "Grade": 525, "Stage": 530
}
assert panel_c["iterations"].eq(N_ITER).all()
assert panel_c["pool_size"].eq(POOL_N).all()
assert panel_c["seed"].eq(SEED).all()
assert np.isfinite(
    panel_c[["observed_rho", "null_mean", "null_sd", "percentile", "empirical_p"]]
).all().all()
assert (panel_c["null_sd"] > 0).all()
assert panel_c["percentile"].between(0, 100).all()
assert panel_c["empirical_p"].between(0, 1).all()
assert len(modules) == N_ITER
assert all(len(module) == len(set(module)) == len(GENES) for module in modules)
panel_c.to_csv(OUTPUT_DIR / "panel_c_locked_matched_null.csv", index=False)
print("Panel C structural audit: PASS")


## Panel D — global-expression PC robustness


In [ ]:
variance = X.var(axis=1, ddof=0)
top_genes = variance.loc[variance > 0].nlargest(5000).index
pca_matrix = X.loc[top_genes].T.to_numpy()
pca_matrix = (
    pca_matrix - pca_matrix.mean(axis=0)
) / pca_matrix.std(axis=0, ddof=0)
pca = PCA(n_components=3, svd_solver="full").fit(pca_matrix)
pcs = pca.transform(pca_matrix)[column_indices]
for index in range(3):
    tcga_ordered[f"PC{index + 1}"] = pcs[:, index]

all_module_scores = z_array[module_indices][:, :, column_indices].mean(axis=1)
observed_all = z_array[target_indices][:, column_indices].mean(axis=0)

pc_rows = []
for count, label in [(1, "PC1"), (3, "PC1-3")]:
    covariates = ["age", "male", "purity"] + [
        f"PC{index}" for index in range(1, count + 1)
    ]
    mask = tcga_ordered[["grade"] + covariates].notna().all(axis=1).to_numpy()
    outcome = tcga_ordered.loc[mask, "grade"].to_numpy(float)
    covariate_matrix = tcga_ordered.loc[mask, covariates].to_numpy(float)
    covariate_matrix = np.column_stack([
        np.ones(len(outcome)), covariate_matrix
    ])
    q_matrix, _ = np.linalg.qr(covariate_matrix)
    outcome_residual = outcome - q_matrix @ (q_matrix.T @ outcome)

    def residualized_betas(score_matrix):
        score_matrix = score_matrix[:, mask].astype(float)
        score_matrix = (
            score_matrix - score_matrix.mean(axis=1, keepdims=True)
        ) / score_matrix.std(axis=1, ddof=0, keepdims=True)
        residuals = score_matrix - (score_matrix @ q_matrix) @ q_matrix.T
        return (residuals @ outcome_residual) / np.sum(residuals ** 2, axis=1)

    null_betas = residualized_betas(all_module_scores)
    observed_beta = residualized_betas(observed_all.reshape(1, -1))[0]
    percentile = float(100 * np.mean(null_betas < observed_beta))
    empirical_p = float(
        (1 + np.sum(null_betas >= observed_beta)) / (len(null_betas) + 1)
    )
    pc_rows.append({
        "adjustment": label,
        "n": int(mask.sum()),
        "observed_beta": float(observed_beta),
        "null_mean": float(null_betas.mean()),
        "null_sd": float(null_betas.std(ddof=1)),
        "percentile": percentile,
        "empirical_p": empirical_p,
    })
    pd.DataFrame({
        "beta_per_sd": null_betas,
        "genes": ["|".join(module) for module in modules],
    }).to_csv(
        OUTPUT_DIR / f"locked_{label.lower().replace('-', '_')}_grade_null.csv",
        index=False,
    )

panel_d_pc = pd.DataFrame(pc_rows)
raw_grade = panel_c.loc[panel_c["endpoint"].eq("Grade")].iloc[0]
panel_d = pd.concat([
    pd.DataFrame([{
        "adjustment": "Raw",
        "n": int(raw_grade["n"]),
        "observed_beta": np.nan,
        "null_mean": raw_grade["null_mean"],
        "null_sd": raw_grade["null_sd"],
        "percentile": raw_grade["percentile"],
        "empirical_p": raw_grade["empirical_p"],
    }]),
    panel_d_pc,
], ignore_index=True)

assert panel_d["adjustment"].tolist() == ["Raw", "PC1", "PC1-3"]
assert np.isfinite(
    panel_d[["null_mean", "null_sd", "percentile", "empirical_p"]]
).all().all()
assert (panel_d["null_sd"] > 0).all()
assert panel_d["percentile"].between(0, 100).all()
assert panel_d["empirical_p"].between(0, 1).all()
assert np.isfinite(panel_d_pc["observed_beta"]).all()
assert np.isfinite(pca.explained_variance_ratio_).all()
assert (pca.explained_variance_ratio_ > 0).all()

panel_d.to_csv(OUTPUT_DIR / "panel_d_global_pc_robustness.csv", index=False)
pd.DataFrame({
    "PC": np.arange(1, 4),
    "variance_explained": pca.explained_variance_ratio_,
}).to_csv(OUTPUT_DIR / "pca_reconstruction.csv", index=False)
print("Panel D structural audit: PASS")


## Final processed-data handoff


In [ ]:
PROCESSED_MANIFEST_PATH = OUTPUT_DIR / "figure_2_processed_manifest.csv"
output_paths = sorted(
    path
    for path in OUTPUT_DIR.glob("*.csv")
    if path != PROCESSED_MANIFEST_PATH
)
manifest_rows = [{
    "file": path.name,
    "path": str(path.resolve()),
    "size_bytes": path.stat().st_size,
    "sha256": sha256_file(path),
} for path in output_paths]
processed_manifest = pd.DataFrame(manifest_rows)
processed_manifest.to_csv(PROCESSED_MANIFEST_PATH, index=False)

print("Figure 2 analysis validation: PASS")
print("Saved outputs to:", OUTPUT_DIR)
display(processed_manifest)
